# Delivery Performance, Delay Risk, and Logistics Efficiency Analysis
## APL Logistics (KWE Group) — Data Analytics Project

**Dataset:** `apl_logistics_analysis_data.csv`  
**Records:** 180,519 | **Fields:** 40 in source data  
**Purpose:** Diagnose delivery timeliness, late-delivery risk, shipping-mode efficiency, geography and customer-segment exposure.

> Privacy note: the supplied data contains customer names and street-level fields. The project analysis file removes direct customer identifiers and street/postal details. Keep the original source file private.
>
> **Important scope limitation:** the supplied CSV has no order date, dispatch date or delivery date. Therefore, a true date-range analysis and time-series trend cannot be produced from this file. The dashboard uses available operational filters instead.

## 1. Imports and load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("apl_logistics_analysis_data.csv.zip", compression="zip", encoding="utf-8-sig")
df.head()

## 2. Data quality and validation

In [ ]:
print("Shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
for col in ["Days for shipping (real)", "Days for shipment (scheduled)", "Late_delivery_risk"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print("Negative shipping durations:", ((df["Days for shipping (real)"] < 0) | (df["Days for shipment (scheduled)"] < 0)).sum())
print("Risk values:", sorted(df["Late_delivery_risk"].dropna().unique()))

## 3. Feature engineering and delivery classification
Delay Gap = actual shipping days − scheduled shipping days. Positive values are delayed, zero is on-time, and negative values are early.

In [ ]:
df["Delay Gap"] = df["Days for shipping (real)"] - df["Days for shipment (scheduled)"]
df["Delivery Classification"] = np.select(
    [df["Delay Gap"] > 0, df["Delay Gap"] < 0],
    ["Delayed", "Early"], default="On-time"
)
summary = df["Delivery Classification"].value_counts().rename_axis("Classification").to_frame("Orders")
summary["Share (%)"] = summary["Orders"] / len(df) * 100
summary

## 4. Baseline KPIs

In [ ]:
kpis = {
    "Total shipments": len(df),
    "On-time exactly (%)": (df["Delay Gap"] == 0).mean() * 100,
    "Delayed (%)": (df["Delay Gap"] > 0).mean() * 100,
    "Early (%)": (df["Delay Gap"] < 0).mean() * 100,
    "Late delivery risk (%)": df["Late_delivery_risk"].mean() * 100,
    "Average delay gap (days)": df["Delay Gap"].mean(),
    "Average delay among delayed shipments (days)": df.loc[df["Delay Gap"] > 0, "Delay Gap"].mean(),
    "Average actual shipping days": df["Days for shipping (real)"].mean(),
    "Average scheduled shipping days": df["Days for shipment (scheduled)"].mean(),
}
pd.Series(kpis).round(2)

## 5. Delay-gap distribution

In [ ]:
df["Delay Gap"].value_counts().sort_index().plot(kind="bar", figsize=(8,4))
plt.xlabel("Actual − scheduled days"); plt.ylabel("Shipment count"); plt.title("Delivery delay-gap distribution"); plt.tight_layout(); plt.show()

## 6. Shipping mode comparison

In [ ]:
mode = df.groupby("Shipping Mode").agg(
    Orders=("Delay Gap","size"),
    Mean_Gap=("Delay Gap","mean"),
    Delayed_Rate=("Delivery Classification", lambda s: (s=="Delayed").mean()*100),
    Risk_Rate=("Late_delivery_risk","mean")
).sort_values("Delayed_Rate", ascending=False)
mode

## 7. Market and regional diagnostics

In [ ]:
market = df.groupby("Market").agg(
    Orders=("Delay Gap","size"),
    Delayed_Rate=("Delivery Classification", lambda s: (s=="Delayed").mean()*100),
    Mean_Gap=("Delay Gap","mean"),
    Risk_Rate=("Late_delivery_risk","mean")
).sort_values("Delayed_Rate", ascending=False)
region = df.groupby("Order Region").agg(
    Orders=("Delay Gap","size"),
    Delayed_Rate=("Delivery Classification", lambda s: (s=="Delayed").mean()*100),
    Mean_Gap=("Delay Gap","mean"),
    Risk_Rate=("Late_delivery_risk","mean")
).sort_values("Delayed_Rate", ascending=False)
display(market.round(2))
display(region.head(15).round(2))

## 8. Customer-segment impact

In [ ]:
segment = df.groupby("Customer Segment").agg(
    Orders=("Delay Gap","size"),
    Delayed_Rate=("Delivery Classification", lambda s: (s=="Delayed").mean()*100),
    Mean_Gap=("Delay Gap","mean"),
    Risk_Rate=("Late_delivery_risk","mean")
)
segment.round(2)

## 9. Delivery status and risk-label reconciliation
The binary risk flag is not identical to a positive delay gap: the 7,754 cancelled shipments contain 4,423 positive-gap records and 3,331 non-positive-gap records. All 98,977 `Late delivery` status rows have risk=1. Interpret risk and delay-gap as separate fields and investigate cancelled orders before SLA reporting.

In [ ]:
pd.crosstab(df["Delivery Status"], df["Delivery Classification"], margins=True), pd.crosstab(df["Delivery Status"], df["Late_delivery_risk"], margins=True)

## 10. Findings, limitations and recommendations

### Findings
- 57.28% of records have actual shipping time above schedule; 18.70% exactly match schedule and 24.02% are early.
- Mean delay gap across all records is +0.57 days; among delayed records it is +1.62 days.
- `Late_delivery_risk=1` appears on 54.83% of records.
- Second Class has the highest delayed share (79.73%) among shipping modes; First Class is 100% delayed, but its mean gap is 1 day and sample size is 27,814.
- Central Africa has the highest delayed share among regions (60.70%) but only 1,677 records; interpret with sample size.
- Market-level delayed shares are tightly grouped (56.81%–57.69%).
- Customer-segment differences are small (57.11%–57.52%).

### Recommendations
1. Review Second Class service design, hand-offs and planned transit assumptions.
2. Investigate First Class schedule feasibility because every observed record has a +1 day gap.
3. Prioritize Central Africa and other elevated regions for lane-level root-cause review, while using minimum sample thresholds.
4. Separate cancelled shipments from completed delivery SLA calculations.
5. Maintain separate operational measures for actual delay gap and the supplied late-risk flag.
6. Add order, dispatch and delivery timestamps plus lane/carrier/exception codes for trend analysis and predictive modelling.

### Limitations
- No date field: no date filter, trend analysis or seasonality assessment.
- No explicit carrier, route, warehouse, customs or exception-reason fields: root causes cannot be attributed.
- Delivery status, risk flag and gap classification are distinct operational definitions.
- Customer and country names include source-specific spelling/locale variants; no aggressive normalization was applied.
- This is descriptive analysis, not causal inference or a predictive model.